In [1]:
import sys
print(sys.executable)

D:\condaData\envs_dirs\ai_agent\python.exe


In [2]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

def get_weather(city: str) -> str:
    """
    查询指定城市的实时天气（使用 OpenWeatherMap API）
    
    参数:
        city: 城市名（中文或英文）
    
    返回:
        天气信息字符串
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")  # 从环境变量读取
    if not api_key:
        return "错误：未配置天气 API 密钥"
    
    # 支持中文城市名，但 API 需要拼音或英文，这里先尝试直接传参
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric&lang=zh_cn"
    
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        if response.status_code != 200:
            return f"查询失败：{data.get('message', '未知错误')}"
        
        weather_desc = data["weather"][0]["description"]
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        return f"{city} 天气：{weather_desc}，温度 {temp}℃，湿度 {humidity}%"
    except Exception as e:
        return f"天气查询出错：{e}"

In [3]:
from langchain_core.tools import tool

@tool
def weather_tool(city: str) -> str:
    """
    查询指定城市的实时天气。
    
    输入格式：城市名称（例如：北京市、上海市）
    """
    return get_weather(city)

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 复用之前的嵌入模型（硅基流动）
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-large-zh-v1.5",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1"
)

# 加载持久化的向量库
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

C:\Users\naizz\AppData\Local\Temp\ipykernel_39924\2381393922.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [5]:
def search_library(query: str, k: int = 3) -> str:
    """
    在图书馆文档中检索与问题相关的信息。
    
    参数:
        query: 用户的问题
        k: 返回的文档块数量
    
    返回:
        检索到的内容（多个块拼接）
    """
    docs = vectorstore.similarity_search(query, k=k)
    if not docs:
        return "未找到相关信息。"
    context = "\n\n".join([doc.page_content for doc in docs])
    return context

In [6]:
@tool
def rag_search_tool(query: str) -> str:
    """
    在图书馆知识库中检索信息。当用户询问图书馆相关规则、开放时间、借阅等问题时使用。
    """
    return search_library(query, k=3)

In [7]:
# 之前定义的工具
from tools import add_tool, multiply_tool  # 假设已导入

# 新工具
tools = [add_tool, multiply_tool, weather_tool, rag_search_tool]


向量库加载成功


In [8]:
from langchain_core.prompts import ChatPromptTemplate

react_prompt_with_memory = ChatPromptTemplate.from_template("""
Answer the following questions as best you can. You have access to the following tools:

{tools}

Previous conversation:
{chat_history}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}
""")

In [9]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI

# 配置 LLM（使用硅基流动）
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 记忆
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Agent
agent = create_react_agent(llm, tools, react_prompt_with_memory)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (C:\Users\naizz\AppData\Roaming\Python\Python313\site-packages\langchain\agents\__init__.py)

In [22]:
# 测试1：知识库问答（图书馆开放时间）
print("=== 测试1：知识库问答 ===")
res1 = agent_executor.invoke({"input": "方闻馆开放时间是什么？"})
print("回答:", res1["output"])
"""
# 测试2：实时天气
print("\n=== 测试2：实时天气 ===")
res2 = agent_executor.invoke({"input": "今天上海天气怎么样？"})
print("回答:", res2["output"])

# 测试3：混合任务（需要记忆+工具+检索）
print("\n=== 测试3：混合任务 ===")
# 先记住名字
agent_executor.invoke({"input": "我叫小明"})
# 然后问复杂问题
res3 = agent_executor.invoke({"input": "我叫什么名字？方闻馆几点关门？今天上海天气如何？帮我算一下 15 + 27"})
print("回答:", res3["output"])
"""

=== 测试1：知识库问答 ===


> Entering new AgentExecutor chain...
用户询问的是图书馆的开放时间，这属于图书馆相关规则的问题，应该使用rag_search_tool来检索信息。

Action: rag_search_tool
Action Input: 方闻馆开放时间一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册        3册
全日制本科生          100册       30册        3册
全日制研究生          100册       30册        3册
非全日制研究生        30册        3册         1册

企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册         1册         1册
继续教育读者          5册         1册         1册

三、方闻馆详细时间
─────────────────────────────────
咨询台：周一至周日 8:30-22:30
东阅览室：周一至周日 8:30-22:30
西阅览室：周一至周日 8:30-22:30
珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）
三层密集书库：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）Final Answer: 方闻馆的开放时间如下：
- 咨询台：周一至周日 8:30-22:30
- 东阅览室：周一至周日 8:30-22:30
- 西阅览室：周一至周日 8:30-22:30
- 珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）
- 三层密集书库：周一

'\n# 测试2：实时天气\nprint("\n=== 测试2：实时天气 ===")\nres2 = agent_executor.invoke({"input": "今天上海天气怎么样？"})\nprint("回答:", res2["output"])\n\n# 测试3：混合任务（需要记忆+工具+检索）\nprint("\n=== 测试3：混合任务 ===")\n# 先记住名字\nagent_executor.invoke({"input": "我叫小明"})\n# 然后问复杂问题\nres3 = agent_executor.invoke({"input": "我叫什么名字？方闻馆几点关门？今天上海天气如何？帮我算一下 15 + 27"})\nprint("回答:", res3["output"])\n'

In [11]:
# 测试1：知识库问答（图书馆开放时间）
print("=== 测试1：知识库问答 ===")
res1 = agent_executor.invoke({"input": "方闻馆开放时间是什么？"})
print("回答:", res1["output"])

=== 测试1：知识库问答 ===


> Entering new AgentExecutor chain...
用户询问的是方闻馆的开放时间，这属于图书馆相关规则的问题，应该使用rag_search_tool进行检索。

Action: rag_search_tool  
Action Input: 方闻馆开放时间  一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册        3册
全日制本科生          100册       30册        3册
全日制研究生          100册       30册        3册
非全日制研究生        30册        3册         1册

企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册         1册         1册
继续教育读者          5册         1册         1册

三、方闻馆详细时间
─────────────────────────────────
咨询台：周一至周日 8:30-22:30
东阅览室：周一至周日 8:30-22:30
西阅览室：周一至周日 8:30-22:30
珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）
三层密集书库：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）Final Answer: 方闻馆各区域开放时间如下：
- 咨询台：周一至周日 8:30-22:30  
- 东阅览室：周一至周日 8:30-22:30  
- 西阅览室：周一至周日 8:30-22:30  
- 珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放） 

In [2]:
from langchain.agents import create_react_agent, AgentExecutor
print("导入成功")

ImportError: cannot import name 'create_react_agent' from 'langchain.agents' (D:\condaData\envs_dirs\ai_agent\Lib\site-packages\langchain\agents\__init__.py)

In [3]:
import sys
print(sys.executable)

D:\condaData\envs_dirs\ai_agent\python.exe


In [4]:
import langchain
print(langchain.__version__)

1.2.13


In [5]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from tools import tools
import os

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 创建 Agent（v1 新语法）
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个乐于助人的助手。"
)

# 调用 Agent
result = agent.invoke({
    "messages": [{"role": "user", "content": "续借规则是什么？"}]
})
print(result["output"])  # 或 result["messages"][-1].content

C:\Users\naizz\ai_first_call\tools.py:37: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


向量库加载成功


BadRequestError: Error code: 400 - {'code': 20015, 'message': '"messages" in request are illegal.', 'data': None}

In [6]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from tools import tools
import os

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个乐于助人的助手。"
)

# 正确的调用方式
result = agent.invoke({"input": "续借规则是什么？"})
print(result["output"])

KeyError: 'output'

In [7]:
result = agent.invoke("续借规则是什么？")

InvalidUpdateError: Expected dict, got 续借规则是什么？
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

In [8]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from tools import tools
import os

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个乐于助人的助手。"
)

# 调用并获取结果
result = agent.invoke({"input": "续借规则是什么？"})

# 打印完整结果，查看结构（调试用）
print("完整结果:", result)

# 从 messages 中提取最后一条 AI 回复
if "messages" in result:
    last_message = result["messages"][-1]
    if hasattr(last_message, "content"):
        answer = last_message.content
    else:
        answer = last_message.get("content", "未找到回答")
    print("答案:", answer)
else:
    print("结果中没有 'messages' 键，请检查返回格式。")

完整结果: {'messages': [AIMessage(content='你好！请问有什么可以帮您的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 355, 'total_tokens': 364, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 355}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V3', 'system_fingerprint': '', 'id': '019d286c76f887b211cd1e6dc85d2f70', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d286c-78fc-7651-a4e8-8ee63cedcdf2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 9, 'total_tokens': 364, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}})]}
答案: 你好！请问有什么可以帮您的吗？


In [9]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from tools import tools   # 你的工具列表
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 创建 Agent（新版）
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个乐于助人的助手，可以调用工具来回答问题。"
)

# 调用 Agent（输入格式：{"messages": [("human", "问题")]}）
# 这是 LangGraph 的标准输入格式
result = agent.invoke({"messages": [("human", "续借规则是什么？")]})

# 打印完整结果（调试用）
print("完整结果:", result)

# 从 messages 中提取最后一条 AI 回复
if "messages" in result:
    last_message = result["messages"][-1]
    # 消息对象可能是 dict 或 BaseMessage 子类
    if isinstance(last_message, dict):
        answer = last_message.get("content", "未找到回答")
    else:
        answer = last_message.content
    print("答案:", answer)
else:
    print("结果中没有 'messages' 键，请检查返回格式。")

BadRequestError: Error code: 400 - {'code': 20015, 'message': '"messages" in request are illegal.', 'data': None}

In [10]:
from langchain.agents.react.agent import create_react_agent
from langchain.agents import AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from tools import tools
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

react_prompt = ChatPromptTemplate.from_template("""
Answer the following questions as best you can. You have access to the following tools:

{tools}

Previous conversation:
{chat_history}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}
""")

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

result = agent_executor.invoke({"input": "续借规则是什么？"})
print(result["output"])

ModuleNotFoundError: No module named 'langchain.agents.react'

In [11]:
from langchain.agents import AgentExecutor, ConversationalChatAgent
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from tools import tools
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 创建对话 Agent
agent = ConversationalChatAgent.from_llm_and_tools(
    llm=llm,
    tools=tools,
    system_message="你是一个乐于助人的助手，可以调用工具来回答问题。"
)

# 创建执行器
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

# 测试
result = agent_executor.invoke({"input": "续借规则是什么？"})
print(result["output"])

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (D:\condaData\envs_dirs\ai_agent\Lib\site-packages\langchain\agents\__init__.py)

In [12]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain_core.messages import HumanMessage, AIMessage
from tools import tools
import os
from dotenv import load_dotenv

load_dotenv()

# 配置 LLM
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 创建带记忆的 Agent（LangGraph 内置记忆管理）
agent = create_react_agent(
    model=llm,
    tools=tools,
    # 可以传入一个提示词
    prompt="你是一个乐于助人的助手，可以调用工具来回答问题。"
)

# 记忆管理（LangGraph 的 Agent 内部支持状态，但我们可以手动维护消息列表）
# 简单起见，这里只演示单次调用，后续可扩展为多轮对话
messages = [("human", "续借规则是什么？")]

# 调用 Agent
result = agent.invoke({"messages": messages})

# 打印结果
print("完整结果:", result)

# 提取最后一条 AI 消息
last_message = result["messages"][-1]
print("答案:", last_message.content)

ModuleNotFoundError: No module named 'langchain.memory'

In [13]:
from langchain_core.memory import ConversationBufferMemory

ModuleNotFoundError: No module named 'langchain_core.memory'

In [14]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from tools import tools
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 创建 Agent
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="你是一个乐于助人的助手，可以调用工具来回答问题。"
)

# 示例调用（无历史）
messages = [HumanMessage(content="续借规则是什么？")]
result = agent.invoke({"messages": messages})
last_message = result["messages"][-1]
print("答案:", last_message.content)

# 如果要维护多轮对话，可以这样：
def chat(question, history_messages):
    messages = history_messages + [HumanMessage(content=question)]
    result = agent.invoke({"messages": messages})
    new_messages = result["messages"]  # 包含所有历史
    return new_messages[-1].content, new_messages

C:\Users\naizz\AppData\Local\Temp\ipykernel_31476\3997189280.py:18: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


BadRequestError: Error code: 400 - {'code': 20015, 'message': '"messages" in request are illegal.', 'data': None}

In [15]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from tools import tools
import os
from dotenv import load_dotenv

load_dotenv()

# 1. 配置 LLM（使用 DeepSeek）
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 2. 创建记忆（新版使用 checkpointer）
memory = MemorySaver()  # 内存版记忆，支持短期多轮对话

# 3. 创建 Agent（新版标准 API）
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个乐于助人的助手，可以调用工具来回答问题。",
    checkpointer=memory  # 开启记忆
)

# 4. 调用 Agent（新版输入格式）
# 每个会话用唯一的 thread_id 来区分
config = {"configurable": {"thread_id": "user_session_001"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "续借规则是什么？"}]},
    config=config
)

# 5. 输出结果
# 新版返回的消息在 messages 列表中，最后一条是 AI 的回复
last_message = result["messages"][-1]
print("答案:", last_message.content)

# 6. 如果需要查看完整执行过程（包括工具调用）
print("\n--- 完整对话历史 ---")
for msg in result["messages"]:
    role = type(msg).__name__
    content = msg.content[:100] if msg.content else "[工具调用]"
    print(f"{role}: {content}")

答案: 续借规则如下：

1. **续借时间**：读者可以在图书到期前进行网上续借。
2. **续借次数**：在最长可外借天数（999天）内，续借次数不限。
3. **限制条件**：
   - 已被预约的图书不能续借。
   - 如果有逾期未还的图书，所有已借图书均不能再办理续借手续。
4. **借阅期限**：
   - 普通流通图书外借期限为40天。
   - 短期外借图书外借期限为14天（不可预约，但可续借1次）。

如果有其他问题，可以随时咨询！

--- 完整对话历史 ---
HumanMessage: 续借规则是什么？
AIMessage: [工具调用]
ToolMessage: 注：已退休但继续承担教学科研任务的教师，凭所在学院证明可申请保留100册借书量。

二、借阅期限
─────────────────────────────────
普通流通图书外借期限：40天
短期
AIMessage: 续借规则如下：

1. **续借时间**：读者可以在图书到期前进行网上续借。
2. **续借次数**：在最长可外借天数（999天）内，续借次数不限。
3. **限制条件**：
   - 已被预约的图书


In [19]:
from langchain_core.tools import tool
print(1)
@tool
def rag_search_tool(query: str) -> str:
    """在图书馆知识库中检索信息。当用户询问图书馆相关规则、开放时间、借阅等问题时使用。"""
    # 你的检索逻辑
    return result

# 工具列表
tools = [multiply_tool, weather_tool, rag_search_tool]

1


NameError: name 'multiply_tool' is not defined

In [17]:
# 同一个 thread_id，多轮对话会自动记忆
config = {"configurable": {"thread_id": "user_001"}}

# 第一轮
result1 = agent.invoke({"messages": [{"role": "user", "content": "我叫小明"}]}, config=config)
print(result1["messages"][-1].content)

# 第二轮（会自动记住名字）
result2 = agent.invoke({"messages": [{"role": "user", "content": "我叫什么名字？"}]}, config=config)
print(result2["messages"][-1].content)  # 应该输出 "小明"

你好，小明！有什么可以帮你的吗？
你刚才告诉我你叫“小明”！需要我帮你做什么吗？
